<a href="https://colab.research.google.com/github/mochwendy/sentimen-app/blob/main/Portofolio_AI_Engineer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Memanggil model ringkasan teks dari Hugging Face
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 2. Masukkan teks panjang yang ingin dirangkum (Contoh teks bahasa Inggris)
teks_panjang = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the intelligence of humans and other animals.
AI applications include advanced web search engines, recommendation systems, understanding human speech, self-driving cars,
and automated decision-making. As machines become increasingly capable, tasks considered to require "intelligence" are often
removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is
frequently excluded from things considered to be AI, having become a routine technology.
"""

# 3. Jalankan proses AI
inputs = tokenizer(teks_panjang, max_length=1024, truncation=True, return_tensors="pt")
summary_ids = model.generate(
    inputs["input_ids"], num_beams=4, max_length=50, min_length=10, early_stopping=True
)
hasil_ringkasan = tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# 4. Tampilkan hasil ringkasan
print("Hasil Ringkasan AI:")
print(hasil_ringkasan)

In [ ]:
from transformers import pipeline

# 1. Ganti tugas menjadi "text-generation" agar sesuai dengan library versi terbaru
perangkum = pipeline("text-generation", model="facebook/bart-large-cnn")

# 2. Masukkan teks panjang yang ingin dirangkum
teks_panjang = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the intelligence of humans and other animals.
AI applications include advanced web search engines, recommendation systems, understanding human speech, self-driving cars,
and automated decision-making. As machines become increasingly capable, tasks considered to require "intelligence" are often
removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is
frequently excluded from things considered to be AI, having become a routine technology.
"""

# 3. Jalankan proses AI (Ganti max_length menjadi max_new_tokens agar kompatibel)
hasil = perangkum(teks_panjang, max_new_tokens=50, do_sample=False)

# 4. Tampilkan hasil ringkasan
print("Hasil Ringkasan AI:")
print(hasil[0]['generated_text'])


In [ ]:
# 1. Panggil komponen spesifik untuk peringkasan teks
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/bart-large-cnn"

# 2. Muat model dan pengolah teks (tokenizer) sesuai arsitektur aslinya
tokenizer = AutoTokenizer.from_pretrained(model_name, clean_up_tokenization_spaces=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Teks panjang yang ingin dirangkum
teks_panjang = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the intelligence of humans and other animals.
AI applications include advanced web search engines, recommendation systems, understanding human speech, self-driving cars,
and automated decision-making. As machines become increasingly capable, tasks considered to require "intelligence" are often
removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is
frequently excluded from things considered to be AI, having become a routine technology.
"""

# 4. Ubah teks menjadi angka (token) yang dipahami oleh AI
inputs = tokenizer(teks_panjang, return_tensors="pt", max_length=1024, truncation=True)

# 5. Jalankan model AI untuk merangkum teks
summary_ids = model.generate(inputs["input_ids"], max_length=50, min_length=10, length_penalty=2.0, num_beams=4, early_stopping=True)

# 6. Ubah kembali angka dari AI menjadi teks manusia yang bersih
hasil_bersih = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Hasil Ringkasan AI yang Benar:")
print(hasil_bersih)


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# 1. Tentukan nama model analisis sentimen dari Hugging Face
nama_model = "distilbert-base-uncased-finetuned-sst-2-english"

# 2. Muat pengolah teks (tokenizer) dan model AI
pengolah_teks = AutoTokenizer.from_pretrained(nama_model)
model_emosi = AutoModelForSequenceClassification.from_pretrained(nama_model)

# 3. MASUKKAN KALIMAT YANG INGIN DIUJI DI SINI (Gunakan Bahasa Inggris)
kalimat_uji = "I am so frustrated because the system keeps crashing and I lost all my unsaved progress."

# 4. Ubah kalimat menjadi format angka yang dipahami AI
input_data = pengolah_teks(kalimat_uji, return_tensors="pt")

# 5. Jalankan AI untuk memprediksi emosi kalimat
with torch.no_grad():
    output = model_emosi(**input_data)

# 6. Hitung hasil skor prediksi
prediksi = torch.nn.functional.softmax(output.logits, dim=-1)
skor_tertinggi = torch.argmax(prediksi).item()
persentase = prediksi[0][skor_tertinggi].item() * 100

# 7. Tampilkan hasil tebakan AI ke layar
label_emosi = "POSITIF (Senang/Pujian)" if skor_tertinggi == 1 else "NEGATIF (Sedih/Kecewa/Marah)"

print("Kalimat Anda:")
print(f'"{kalimat_uji}"\n')
print(f"Hasil Analisis AI: Kalimat ini bermakna {label_emosi}")
print(f"Tingkat Keyakinan AI: {persentase:.2f}%")


In [ ]:
!pip install streamlit transformers torch -q
!npm install -g localtunnel -q


In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

st.set_page_config(page_title="AI Sentiment Analyzer", page_icon="🤖")
st.title("🤖 AI Pendeteksi Emosi Teks")
st.write("Masukkan kalimat dalam Bahasa Inggris untuk menganalisis apakah maknanya Positif atau Negatif.")

@st.cache_resource
def load_model():
    nama_model = "distilbert-base-uncased-finetuned-sst-2-english"
    pengolah_teks = AutoTokenizer.from_pretrained(nama_model)
    model_emosi = AutoModelForSequenceClassification.from_pretrained(nama_model)
    return pengolah_teks, model_emosi

pengolah_teks, model_emosi = load_model()

kalimat_uji = st.text_area("Tulis kalimat Anda di sini:", "I am very happy to learn AI engineering today!")

if st.button("Analisis Sentimen"):
    with st.spinner("AI sedang berpikir..."):
        input_data = pengolah_teks(kalimat_uji, return_tensors="pt")
        with torch.no_grad():
            output = model_emosi(**input_data)

        prediksi = torch.nn.functional.softmax(output.logits, dim=-1)
        skor_tertinggi = torch.argmax(prediksi).item()
        persentase = prediksi[skor_tertinggi].item() * 100

        st.write("---")
        if skor_tertinggi == 1:
            st.success(f"**Hasil:** POSITIF (Senang/Pujian) 🎉")
        else:
            st.error(f"**Hasil:** NEGATIF (Sedih/Kecewa/Marah) ⚠️")

        st.info(f"**Tingkat Keyakinan AI:** {persentase:.2f}%")



In [ ]:
!wget -qO- ://icanhazip.com


In [ ]:
!wget -qO- ://icanhazip.com

In [ ]:
!curl ipecho.net/plain; echo
!streamlit run app.py & npx localtunnel --port 8501



In [ ]:
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 py@pinggy.io


In [ ]:
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 py@pinggy.io



In [ ]:
!ssh -p 443 -o StrictHostKeyChecking=no -R 80:localhost:8501 free@a.pinggy.io


In [ ]:
import urllib.request
import json

# Mengambil info tautan aktif dari lokal server Pinggy
try:
    with urllib.request.urlopen("http://localhost:4321/urls") as url:
        data = json.loads(url.read().decode())
        print("🎉 BERHASIL! Klik tautan di bawah ini untuk membuka Web AI Anda:")
        print("----------------------------------------------------------------")
        print(data['urls'][0])
        print("----------------------------------------------------------------")
except Exception:
    print("Gagal mengambil link otomatis. Silakan cek kembali kotak kode SSH di atas,")
    print("biasanya ada link berakhiran '.pinggy.link' yang bisa langsung diklik.")


In [ ]:
from google.colab import output
import time

# 1. Jalankan server Streamlit di latar belakang pada port 8501
get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

# 2. Tunggu 3 detik agar server Streamlit siap
time.sleep(3)

# 3. Tampilkan antarmuka web Streamlit langsung di dalam halaman Colab
output.serve_kernel_port_as_iframe(8501, height='600')


In [ ]:
# 1. Matikan paksa semua proses streamlit yang menggantung sebelumnya
!pkill streamlit
import time

# 2. Jalankan ulang Streamlit dengan konfigurasi headless (tanpa browser lokal)
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true &')

# 3. Berikan waktu lebih lama (5 detik) agar server benar-benar siap
print("Sedang menyalakan server Streamlit, mohon tunggu...")
time.sleep(5)

# 4. Tampilkan kembali jendela aplikasi web
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height='600')


In [ ]:
!streamlit run app.py --server.port 8501


In [ ]:
from google.colab import output
import time

# 1. Jalankan server Streamlit di port baru (8502)
get_ipython().system_raw('streamlit run app.py --server.port 8502 --server.headless true &')

# 2. Tunggu 5 detik agar server siap
print("Sedang mengalihkan ke Port 8502, mohon tunggu...")
time.sleep(5)

# 3. Tampilkan aplikasi web dari port 8502
output.serve_kernel_port_as_iframe(8502, height='600')


In [ ]:
from pyngrok import ngrok
import time

# 1. Masukkan token autentikasi Anda ke dalam sistem Ngrok
ngrok.set_auth_token("3FZJvkVJemqrHr8Wuko2P26zVgz")

# 2. Bersihkan sisa proses lama & jalankan ulang Streamlit di latar belakang
!pkill streamlit
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true &')

# 3. Beri waktu 3 detik agar server Streamlit siap sepenuhnya
print("Sedang menyiapkan jalur server, mohon tunggu...")
time.sleep(3)

# 4. Buka terowongan publik Ngrok untuk meneruskan port 8501
try:
    # Putus koneksi lama jika ada yang menggantung
    ngrok.disconnect(ngrok.connect(8501).public_url)
except:
    pass

tautan_web = ngrok.connect(8501)

# 5. Cetak tautan resmi untuk Anda klik
print("\n🎉 BERHASIL! Klik tautan resmi di bawah ini untuk membuka Web AI Anda:")
print("---------------------------------------------------------------------")
print(tautan_web.public_url)
print("---------------------------------------------------------------------")
print("Catatan: Jika muncul halaman peringatan dari Ngrok, cukup klik tombol 'Visit Site'.")


In [ ]:
# 1. Install library pyngrok langsung di awal
!pip install pyngrok -q

from pyngrok import ngrok
import time

# 2. Masukkan token autentikasi Anda
ngrok.set_auth_token("3FZJvkVJemqrHr8Wuko2P26zVgz")

# 3. Bersihkan sisa proses lama & jalankan ulang Streamlit di latar belakang
!pkill streamlit
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true &')

# 4. Beri waktu 3 detik agar server Streamlit siap sepenuhnya
print("Sedang menyiapkan jalur server, mohon tunggu...")
time.sleep(3)

# 5. Buka terowongan publik Ngrok untuk meneruskan port 8501
try:
    ngrok.kill() # Membersihkan sisa koneksi terowongan ngrok terdahulu
except:
    pass

tautan_web = ngrok.connect(8501)

# 6. Cetak tautan resmi untuk Anda klik
print("\n🎉 BERHASIL! Klik tautan resmi di bawah ini untuk membuka Web AI Anda:")
print("---------------------------------------------------------------------")
print(tautan_web.public_url)
print("---------------------------------------------------------------------")
print("Catatan: Jika muncul halaman peringatan dari Ngrok, cukup klik tombol 'Visit Site'.")


In [ ]:
# 1. Pastikan library pyngrok terinstal sempurna
!pip install pyngrok -q

from pyngrok import ngrok
import time

# 2. Masukkan token autentikasi resmi Anda yang baru
ngrok.set_auth_token("3FZKaN9J5groF8WYi9Lr7RUBHPK_6b4YHHR6wwvAaPLTjftj5")

# 3. Matikan semua sisa proses lama agar port tidak tersumbat
!pkill streamlit
try:
    ngrok.kill()
except:
    pass

# 4. Jalankan server Streamlit di latar belakang (Port 8501)
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true &')

# 5. Beri jeda 3 detik untuk aktivasi sistem
print("Sedang menyambungkan terowongan Ngrok, mohon tunggu...")
time.sleep(3)

# 6. Buka jalur publik menggunakan protokol HTTP resmi
tautan_web = ngrok.connect(8501, proto="http")

# 7. Cetak link web akhir untuk Anda
print("\n🎉 SUKSES BESAR! Web AI Anda telah online di seluruh dunia:")
print("---------------------------------------------------------------------")
print(tautan_web.public_url)
print("---------------------------------------------------------------------")
print("Catatan: Jika muncul halaman 'ngrok: Welcome', cukup klik tombol 'Visit Site'.")


In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# 1. Judul Aplikasi Web
st.set_page_config(page_title="AI Sentiment Analyzer", page_icon="🤖")
st.title("🤖 AI Pendeteksi Emosi Teks")
st.write("Masukkan kalimat dalam Bahasa Inggris untuk menganalisis apakah maknanya Positif atau Negatif.")

# 2. Load Model AI (Gunakan cache)
@st.cache_resource
def load_model():
    nama_model = "distilbert-base-uncased-finetuned-sst-2-english"
    pengolah_teks = AutoTokenizer.from_pretrained(nama_model)
    model_emosi = AutoModelForSequenceClassification.from_pretrained(nama_model)
    return pengolah_teks, model_emosi

pengolah_teks, model_emosi = load_model()

# 3. Membuat Input Teks di Web
kalimat_uji = st.text_area("Tulis kalimat Anda di sini:", "I am very happy to learn AI engineering today!")

# 4. Membuat Tombol Jalankan
if st.button("Analisis Sentimen"):
    with st.spinner("AI sedang berpikir..."):
        # Jalankan Proses AI
        input_data = pengolah_teks(kalimat_uji, return_tensors="pt")
        with torch.no_grad():
            output = model_emosi(**input_data)

        # Hitung Hasil Skor Prediksi (Ditambahkan [0] untuk mengambil dimensi pertama)
        prediksi = torch.nn.functional.softmax(output.logits, dim=-1)[0]
        skor_tertinggi = torch.argmax(prediksi).item()
        persentase = prediksi[skor_tertinggi].item() * 100

        # Tampilkan Hasil di Web
        st.write("---")
        if skor_tertinggi == 1:
            st.success(f"**Hasil:** POSITIF (Senang/Pujian) 🎉")
        else:
            st.error(f"**Hasil:** NEGATIF (Sedih/Kecewa/Marah) ⚠️")

        st.info(f"**Tingkat Keyakinan AI:** {persentase:.2f}%")


In [ ]:
%%writefile aturan_kantor.txt
Aturan Resmi PT AI CINTA KARYA KITA:
1. Jam kerja kantor dimulai tepat pukul 08.00 WIB dan berakhir pada pukul 17.00 WIB.
2. Setiap karyawan berhak mendapatkan jatah cuti tahunan sebanyak 12 hari kerja.
3. Fasilitas makan siang gratis disediakan oleh perusahaan khusus pada hari Jumat.
4. Pakaian pada hari Senin sampai Kamis adalah kemeja rapi, sedangkan hari Jumat diperbolehkan memakai baju batik atau kaos berkerah.


In [ ]:
import gradio as gr
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Memuat dokumen internal yang sudah dibuat
loader = TextLoader("aturan_kantor.txt", encoding="utf-8")
dokumen = loader.load()

# 2. Memotong teks panjang menjadi bagian-bagian kecil agar mudah dipahami AI
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
potongan_teks = text_splitter.split_documents(dokumen)

# 3. Menggunakan model Hugging Face khusus bahasa Indonesia untuk menyamakan arti kata (Embeddings)
print("Sedang memuat model kecerdasan bahasa, mohon tunggu...")
model_embedding = HuggingFaceEmbeddings(model_name="indobenchmark/indobert-base-p2")

# 4. Menyimpan pengetahuan dokumen ke dalam Database Vektor (ChromaDB) di memori komputer
database_vektor = Chroma.from_documents(potongan_teks, model_embedding)

# 5. Fungsi Utama RAG untuk mencari jawaban paling cocok di dalam dokumen
def tanya_ai_rag(pertanyaan_user):
    # Mencari potongan dokumen yang paling mirip dengan pertanyaan
    hasil_pencarian = database_vektor.similarity_search(pertanyaan_user, k=1)

    if hasil_pencarian:
        return f"📖 Hasil Temuan AI di Dokumen:\n\n{hasil_pencarian[0].page_content}"
    else:
        return "Maaf, informasi tersebut tidak ditemukan dalam dokumen internal."

# 6. Membuat Tampilan Web Interaktif dengan Gradio
antarmuka_rag = gr.Interface(
    fn=tanya_ai_rag,
    inputs=gr.Textbox(lines=2, placeholder="Contoh: Jam berapa kantor dimulai? / Berapa hari jatah cuti?", label="Pertanyaan Anda"),
    outputs=gr.Textbox(label="Jawaban AI berdasarkan Dokumen"),
    title="🏢 AI Asisten Dokumen Perusahaan (RAG)",
    description="AI ini akan menjawab pertanyaan Anda murni berdasarkan dokumen aturan internal perusahaan."
)

# 7. Jalankan web dan buat tautan publik otomatis
antarmuka_rag.launch(share=True, debug=True)


In [ ]:
# 1. Install ulang seluruh library pendukung secara paksa dan eksplisit
!pip install langchain langchain-community langchain-text-splitters chromadb pypdf gradio sentence-transformers -q

import gradio as gr
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

print("✅ Semua library berhasil dimuat!")

# 2. Pastikan file aturan_kantor.txt ada
if not os.path.exists("aturan_kantor.txt"):
    with open("aturan_kantor.txt", "w", encoding="utf-8") as f:
        f.write("""Aturan Resmi PT AI Maju Bersama:
1. Jam kerja kantor dimulai tepat pukul 08.00 WIB dan berakhir pada pukul 17.00 WIB.
2. Setiap karyawan berhak mendapatkan jatah cuti tahunan sebanyak 12 hari kerja.
3. Fasilitas makan siang gratis disediakan oleh perusahaan khusus pada hari Jumat.
4. Pakaian pada hari Senin sampai Kamis adalah kemeja rapi, sedangkan hari Jumat diperbolehkan memakai baju batik atau kaos berkerah.""")

# 3. Memuat dan memotong dokumen internal
loader = TextLoader("aturan_kantor.txt", encoding="utf-8")
dokumen = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
potongan_teks = text_splitter.split_documents(dokumen)

# 4. Memuat model kecerdasan bahasa Indonesia (Embeddings)
print("⏳ Sedang mengunduh model bahasa Indonesia (indobert), mohon tunggu...")
model_embedding = HuggingFaceEmbeddings(model_name="indobenchmark/indobert-base-p2")

# 5. Menyimpan ke Database Vektor (ChromaDB)
database_vektor = Chroma.from_documents(potongan_teks, model_embedding)

# 6. Fungsi Utama pencarian dokumen RAG
def tanya_ai_rag(pertanyaan_user):
    hasil_pencarian = database_vektor.similarity_search(pertanyaan_user, k=1)
    if hasil_pencarian:
        return f"📖 Hasil Temuan AI di Dokumen:\n\n{hasil_pencarian[0].page_content}"
    else:
        return "Maaf, informasi tersebut tidak ditemukan dalam dokumen internal."

# 7. Membuat dan Menjalankan Tampilan Web dengan Gradio
antarmuka_rag = gr.Interface(
    fn=tanya_ai_rag,
    inputs=gr.Textbox(lines=2, placeholder="Contoh: Jam berapa kantor dimulai? / Hari apa ada makan siang gratis?", label="Pertanyaan Anda"),
    outputs=gr.Textbox(label="Jawaban AI berdasarkan Dokumen"),
    title="🏢 AI Asisten Dokumen Perusahaan (RAG Bahasa Indonesia)",
    description="AI ini akan menjawab pertanyaan Anda murni berdasarkan dokumen aturan internal perusahaan."
)

# Jalankan web dan buat tautan publik otomatis
antarmuka_rag.launch(share=True, debug=True)


In [ ]:
from pyngrok import ngrok
# Mematikan semua terowongan data yang masih aktif
ngrok.kill()
print("🔒 Semua jalur terowongan Ngrok telah ditutup dengan aman!")


In [ ]:
# 1. Pastikan library pembaca PDF terbaru terpasang
!pip install pypdf chromadb sentence-transformers gradio -q

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
import os

# 2. Fungsi membaca file PDF asli dan memotongnya per halaman/paragraf
def ekstraksi_pdf(path_pdf):
    reader = PdfReader(path_pdf)
    potongan_teks = []
    for i, halaman in enumerate(reader.pages):
        teks_halaman = halaman.extract_text()
        if teks_halaman.strip():
            # Tambahkan penanda halaman agar informasi valid
            potongan_teks.append(f"[Halaman {i+1}] {teks_halaman.strip()}")
    return potongan_teks

# Jalankan ekstraksi jika file PDF tersedia
if os.path.exists("dokumen_kita.pdf"):
    data_pdf = ekstraksi_pdf("dokumen_kita.pdf")
else:
    # Contoh data darurat jika user belum upload PDF
    data_pdf = ["Silakan unggah file bernama 'dokumen_kita.pdf' ke panel kiri Google Colab Anda."]

# 3. Inisialisasi Model AI Bahasa Indonesia & Database Vektor
model_ai = SentenceTransformer("indobenchmark/indobert-base-p2")
klien_chroma = chromadb.Client()

try:
    koleksi = klien_chroma.create_collection(name="rag_pdf_indonesia")
except:
    klien_chroma.delete_collection(name="rag_pdf_indonesia")
    koleksi = klien_chroma.create_collection(name="rag_pdf_indonesia")

# Masukkan data PDF ke database
vektor_pdf = model_ai.encode(data_pdf).tolist()
koleksi.add(
    embeddings=vektor_pdf,
    documents=data_pdf,
    ids=[f"id_{i}" for i in range(len(data_pdf))]
)

# 4. Fungsi Pencarian RAG
def tanya_pdf_ai(pertanyaan):
    vektor_tanya = model_ai.encode([pertanyaan]).tolist()
    hasil = koleksi.query(query_embeddings=vektor_tanya, n_results=1)
    if hasil and hasil['documents']:
        return f"🔍 Hasil Analisis Dokumen PDF:\n\n{hasil['documents'][0][0]}"
    return "Informasi tidak ditemukan di dalam PDF."

# 5. Jalankan Web Interaktif
web_rag = gr.Interface(
    fn=tanya_pdf_ai,
    inputs=gr.Textbox(placeholder="Apa yang ingin Anda tanyakan dari file PDF ini?", label="Pertanyaan User"),
    outputs=gr.Textbox(label="Jawaban Hasil Scan AI"),
    title="📂 AI Intelligent PDF Reader (RAG)"
)
web_rag.launch(share=True, debug=True)


In [ ]:
# PERBAIKAN: Mengubah fungsi pencarian agar mengambil 3 potongan halaman teratas
def tanya_pdf_ai_v2(pertanyaan):
    vektor_tanya = model_ai.encode([pertanyaan]).tolist()

    # n_results dinaikkan menjadi 3 agar informasi halaman sekitar ikut terbaca
    hasil = koleksi.query(query_embeddings=vektor_tanya, n_results=3)

    if hasil and hasil['documents']:
        # Menggabungkan seluruh teks temuan agar menjadi bacaan yang utuh
        gabungan_jawaban = "\n\n---\n\n".join(hasil['documents'][0])
        return f"🔍 Hasil Analisis Dokumen PDF (Multi-Halaman):\n\n{gabungan_jawaban}"
    return "Informasi tidak ditemukan di dalam PDF."

# Jalankan ulang antarmuka web dengan fungsi baru
web_rag_v2 = gr.Interface(
    fn=tanya_pdf_ai_v2,
    inputs=gr.Textbox(placeholder="Apa saja pengalaman kerja Mochamad Wendy?", label="Pertanyaan User"),
    outputs=gr.Textbox(label="Jawaban Hasil Scan AI"),
    title="📂 AI Intelligent PDF Reader v2 (RAG)"
)
web_rag_v2.launch(share=True, debug=True)


In [ ]:
# 1. Pastikan library penghubung LLM resmi terpasang
!pip install groq langchain-core -q

import os
import gradio as gr
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

# 2. Inisialisasi token API Groq resmi milik Anda
os.environ["GROQ_API_KEY"] = "3FZRVahpfUpBbiNP3g64HIGXxTn_7pWb5SCennZGarLMbrkKx"
klien_groq = Groq()

# Muat kembali model pencarian teks (Embeddings) untuk database
model_ai = SentenceTransformer("indobenchmark/indobert-base-p2")
klien_chroma = chromadb.Client()
koleksi = klien_chroma.get_collection(name="rag_pdf_indonesia")

# 3. Fungsi Utama True RAG: Menghubungkan Pencarian Dokumen dengan Penalaran LLM
def tanya_pdf_dengan_llm(pertanyaan_user):
    # A. Cari 3 potongan halaman paling relevan dari file PDF di database vektor
    vektor_tanya = model_ai.encode([pertanyaan_user]).tolist()
    hasil_database = koleksi.query(query_embeddings=vektor_tanya, n_results=3)

    # Gabungkan dokumen temuan menjadi satu teks pendukung
    teks_pendukung = "\n\n---\n\n".join(hasil_database['documents'][0])

    # B. Susun prompt instruksi ketat (System Prompt) agar AI patuh pada isi PDF
    prompt_instruksi = f"""
    Anda adalah AI Asisten Dokumen Profesional yang cerdas. Tugas Anda adalah menjawab pertanyaan user
    HANYA berdasarkan informasi dari Teks Pendukung Dokumen yang dilampirkan di bawah ini.

    Jawablah dengan bahasa Indonesia yang sangat rapi, sopan, dan langsung ke inti jawaban.
    Jika pertanyaan user menanyakan tentang 'project' atau 'proyek', artikan itu sebagai portfolio,
    aplikasi, tugas akhir, atau hasil kerja nyata yang tertera pada dokumen.
    Jika jawabannya sama sekali tidak tertera di dokumen, katakan secara jujur: 'Maaf, data tersebut tidak ditemukan dalam dokumen PDF Anda.'

    Teks Pendukung Dokumen:
    {teks_pendukung}

    Pertanyaan User: {pertanyaan_user}
    Jawaban Cerdas Anda:
    """

    # C. Alirkan teks ke otak LLM Llama 3 untuk melahirkan jawaban cerdas manusiawi
    respons = klien_groq.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt_instruksi}],
        temperature=0.2 # Nilai rendah agar AI tetap patuh pada data & tidak mengarang bebas (halusinasi)
    )

    return respons.choices.message.content

# 4. Tampilkan Antarmuka Web Premium dengan Gradio
web_true_rag = gr.Interface(
    fn=tanya_pdf_dengan_llm,
    inputs=gr.Textbox(lines=2, placeholder="Contoh: Project apa saja yang sudah dikerjakan Mochamad Wendy?", label="Pertanyaan User"),
    outputs=gr.Textbox(label="Jawaban Analisis Cerdas LLM (Llama 3)"),
    title="🤖 AI Intelligent PDF Reader v3 (True RAG)",
    description="Aplikasi RAG standar industri. Menggabungkan pencarian presisi ChromaDB dengan otak penalaran Llama 3 via Groq API."
)

# Jalankan web dan buat tautan publik baru
web_true_rag.launch(share=True, debug=True)


In [ ]:
# PERBAIKAN FINAL: Menambahkan [0] pada hasil_database['documents']
def tanya_pdf_dengan_llm_v2(pertanyaan_user):
    # A. Cari 3 potongan halaman paling relevan
    vektor_tanya = model_ai.encode([pertanyaan_user]).tolist()
    hasil_database = koleksi.query(query_embeddings=vektor_tanya, n_results=3)

    # PERBAIKAN DI SINI: Tambahkan [0] agar Python membaca teks di dalam list
    teks_pendukung = "\n\n---\n\n".join(hasil_database['documents'][0])

    # B. Susun prompt instruksi ketat untuk Llama 3
    prompt_instruksi = f"""
    Anda adalah AI Asisten Dokumen Profesional yang cerdas. Tugas Anda adalah menjawab pertanyaan user
    HANYA berdasarkan informasi dari Teks Pendukung Dokumen yang dilampirkan di bawah ini.

    Jawablah dengan bahasa Indonesia yang sangat rapi, sopan, dan langsung ke inti jawaban.
    Jika pertanyaan user menanyakan tentang 'project' atau 'proyek', artikan itu sebagai portfolio,
    aplikasi, tugas akhir, atau hasil kerja nyata yang tertera pada dokumen.
    Jika jawabannya sama sekali tidak tertera di dokumen, katakan secara jujur: 'Maaf, data tersebut tidak ditemukan dalam dokumen PDF Anda.'

    Teks Pendukung Dokumen:
    {teks_pendukung}

    Pertanyaan User: {pertanyaan_user}
    Jawaban Cerdas Anda:
    """

    # C. Kirim ke Llama 3 via Groq API
    respons = klien_groq.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt_instruksi}],
        temperature=0.2
    )

    return respons.choices.message.content

# Jalankan ulang antarmuka web Gradio
web_true_rag_v2 = gr.Interface(
    fn=tanya_pdf_dengan_llm_v2,
    inputs=gr.Textbox(lines=2, placeholder="Project apa saja yang sudah dikerjakan Mochamad Wendy?", label="Pertanyaan User"),
    outputs=gr.Textbox(label="Jawaban Analisis Cerdas LLM (Llama 3)"),
    title="🤖 AI Intelligent PDF Reader v3.1 (True RAG Fixed)"
)
web_true_rag_v2.launch(share=True, debug=True)


In [ ]:
import gradio as gr
import chromadb
from transformers import pipeline
import torch

print("⏳ Sedang menyiapkan otak LLM Bahasa Indonesia, mohon tunggu...")

# 1. Panggil LLM Bahasa Indonesia gratis langsung dari Hugging Face
# Menggunakan pipeline text-generation yang stabil
generator_llm = pipeline(
    "text-generation",
    model="LexiLLM/Llama-3-8B-Lexi-Indonesia-v1",
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. Fungsi Utama True RAG Lokal
def tanya_pdf_lokal(pertanyaan_user):
    # A. Ambil dokumen dari database vektor ChromaDB yang sudah ada
    vektor_tanya = model_ai.encode([pertanyaan_user]).tolist()
    hasil_database = koleksi.query(query_embeddings=vektor_tanya, n_results=2)
    teks_pendukung = "\n\n".join(hasil_database['documents'][0])

    # B. Buat template percakapan (Prompt) dalam Bahasa Indonesia
    format_prompt = f"""<|im_start||system|
    Anda adalah AI Asisten PDF yang jujur dan cerdas. Jawab pertanyaan berdasarkan dokumen yang diberikan.
    Jika tidak ada di dokumen, katakan tidak ada.<|im_end|>
    <|im_start|>user
    Dokumen Referensi:
    {teks_pendukung}

    Pertanyaan: {pertanyaan_user}<|im_end|>
    <|im_start|>assistant
    """

    # C. Jalankan LLM untuk merangkum jawaban
    respons = generator_llm(format_prompt, max_new_tokens=150, do_sample=False)

    # Ambil teks jawaban saja dari output
    teks_mentah = respons[0]['generated_text']
    jawaban_ai = teks_mentah.split("<|im_start|>assistant")[-1].strip()

    return jawaban_ai

# 3. Tampilkan Aplikasi Web Baru
web_rag_lokal = gr.Interface(
    fn=tanya_pdf_lokal,
    inputs=gr.Textbox(lines=2, placeholder="Project apa saja yang sudah dikerjakan Mochamad Wendy?", label="Pertanyaan User"),
    outputs=gr.Textbox(label="Jawaban Hasil Ekstraksi Pintar AI"),
    title="🤖 AI Intelligent PDF Reader v4 (Local True RAG)"
)

# Jalankan dengan link publik otomatis
web_rag_lokal.launch(share=True, debug=True)


In [ ]:
# 1. Pastikan library pengolahan teks terbaru siap
!pip install gradio transformers torch chromadb -q

import gradio as gr
import chromadb
from transformers import pipeline
import torch

print("⏳ Sedang mengunduh otak LLM Qwen 2.5 (Bebas Akses), mohon tunggu...")

# 2. Muat LLM Qwen yang sepenuhnya terbuka dan mendukung Bahasa Indonesia
generator_llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Fungsi Utama True RAG Lokal tanpa Token API (PERBAIKAN)
def tanya_pdf_lokal_qwen(pertanyaan_user):
    # A. Ambil potongan dokumen asli dari database vektor ChromaDB sebelumnya
    vektor_tanya = model_ai.encode([pertanyaan_user]).tolist()
    hasil_database = koleksi.query(query_embeddings=vektor_tanya, n_results=2)
    teks_pendukung = "\n\n".join(hasil_database['documents'][0])

    # B. Susun struktur percakapan resmi (Chat Template) untuk Qwen
    percakapan = [
        {"role": "system", "content": "Anda adalah AI Asisten PDF yang jujur. Jawab pertanyaan user HANYA berdasarkan dokumen referensi yang diberikan secara lengkap. Jika tidak ada di dokumen, katakan secara jujur bahwa data tidak ditemukan."},
        {"role": "user", "content": f"Dokumen Referensi:\n{teks_pendukung}\n\nPertanyaan: {pertanyaan_user}"}
    ]

    # Mengubah instruksi menjadi format teks khusus yang dipahami Qwen
    prompt_bersih = generator_llm.tokenizer.apply_chat_template(percakapan, tokenize=False, add_generation_prompt=True)

    # C. Jalankan pemrosesan teks (TAMBAHKAN max_new_tokens agar jawaban lengkap)
    respons = generator_llm(
        prompt_bersih,
        do_sample=False,
        max_new_tokens=512, # Memperpanjang kuota jawaban hingga 512 token baru
        clean_up_tokenization_spaces=False # Memperbaiki teks spasi simbol otomatis
    )

    # D. Ambil murni hasil jawaban AI tanpa menyertakan kembali teks perintahnya
    teks_hasil = respons[0]['generated_text']
    jawaban_final = teks_hasil.split("<|im_start|>assistant\n")[-1].strip()

    return jawaban_final


# 4. Bangun Antarmuka Web Tampilan Baru
web_rag_qwen = gr.Interface(
    fn=tanya_pdf_lokal_qwen,
    inputs=gr.Textbox(lines=2, placeholder="Project atau aplikasi apa saja yang sudah dibuat Mochamad Wendy?", label="Pertanyaan Anda"),
    outputs=gr.Textbox(label="Jawaban Hasil Ekstraksi Cerdas LLM"),
    title="🤖 AI Intelligent PDF Reader v4 (True RAG Qwen Open Source)",
    description="Sistem RAG mandiri bebas token. Menggabungkan pencarian ChromaDB dengan penalaran pintar model Qwen 2.5."
)

# Jalankan server dan terbitkan tautan publik otomatis
web_rag_qwen.launch(share=True, debug=True)
